# Gold Feedback Sentiment — PySpark Equivalent

PySpark reimplementation of the Lakeflow Designer pipeline `meeting_gold_feedback_sentiment -  NA`: filter free-form survey answers with AND logic, add AI sentiment on the answer, join session attributes from the sessions side, and overwrite the gold table.

In [0]:
from pyspark.sql.functions import col, expr

catalog = "learn_no_code_etl_designer_puli_reddy"
schema = "designer_meeting"

feedback_silver_table = spark.table(f"{catalog}.{schema}.feedback_silver")
sessions_silver_table = spark.table(f"{catalog}.{schema}.sessions_silver")

free_form_question_ids = [
    "r6Wd31x8THycYxzVzeZ0BA",
    "-TYqDeIsREmunUvAmNrpPw",
    "is7w6QM0RtK1Y3RsL_o8Vg",
]

# Keep only the three free-form questions with non-empty answers.
# All three conditions must hold (AND) — an OR version keeps every row that
# has any answer, i.e. the entire 9,106-row table including rating questions.
feedback_filtered = feedback_silver_table.filter(
    col("question_id").isin(free_form_question_ids)
    & col("answer").isNotNull()
    & (col("answer") != "")
)

feedback_total = feedback_silver_table.count()
sessions_total = sessions_silver_table.count()
filtered_total = feedback_filtered.count()
print(f"feedback_silver rows:    {feedback_total}")
print(f"sessions_silver rows:    {sessions_total}")
print(f"filtered sentiment rows: {filtered_total} (expected 3984)")
assert filtered_total == 3984, f"Expected 3984 filtered rows, got {filtered_total}"

display(feedback_filtered.groupBy("question").count().orderBy("count", ascending=False))

feedback_silver rows:    9106
sessions_silver rows:    41
filtered sentiment rows: 3984 (expected 3984)


question,count
What did you like about the content in this session?,1858
What could have been improved?,1123
What additional topics would you like to get a deep dive on?,1003


In [0]:
# Add AI sentiment on the answer, keeping all original columns.
# ai_analyze_sentiment returns 'positive', 'negative', 'neutral', or 'mixed'.
feedback_sentiment_transform = feedback_filtered.withColumn(
    "sentiment", expr("ai_analyze_sentiment(answer)")
)

display(feedback_sentiment_transform.select("question", "answer", "sentiment").limit(10))

question,answer,sentiment
What did you like about the content in this session?,I think finally understand how all the pieces fit together end to end,positive
What could have been improved?,"I have to say, the session assumed too much prior knowledge of the underlying infrastructure My only complaint is that it went by too fast.",mixed
What did you like about the content in this session?,"To be frank, great real-world examples that connected theory to practice Looking forward to more sessions like this.",positive
What did you like about the content in this session?,The analogies used to explain the architecture were spot on Will be sharing the notebooks with my team.,positive
What could have been improved?,"From my perspective, could use more documentation links for follow-up reading My only complaint is that it went by too fast.",mixed
What did you like about the content in this session?,"To be frank, the performance benchmarks shown were really convincing My only complaint is that it went by too fast.",mixed
What additional topics would you like to get a deep dive on?,Handling schema evolution and schema enforcement at scale,neutral
What did you like about the content in this session?,"To be frank, great pace, not too fast and not too slow Looking forward to more sessions like this.",positive
What did you like about the content in this session?,This session clarified several misconceptions i had about the platform Would definitely attend again.,positive
What could have been improved?,Providing sample datasets ahead of time would help with the hands-on portion This is exactly what our team needed.,positive


In [0]:
# Session attributes to attach (meeting_uuid is the join key).
session_attributes = sessions_silver_table.select(
    "meeting_uuid",
    "event_date",
    "difficulty_level",
    "session_name",
    "planned_duration_minutes",
    "presenter",
    "topic_category",
)

# LEFT join with the sentiment DataFrame on the LEFT so every sentiment row is
# preserved. Sessions on the left would add one extra row: one of the 41
# sessions has no free-form feedback at all.
gold_feedback_sentiment = feedback_sentiment_transform.join(
    session_attributes, on="meeting_uuid", how="left"
).select(
    "event_date",
    "difficulty_level",
    "session_name",
    "meeting_uuid",
    "planned_duration_minutes",
    "presenter",
    "topic_category",
    "sentiment",
    "respondent_id",
    "answer",
    "question",
)

gold_count = gold_feedback_sentiment.count()
print(f"gold_feedback_sentiment rows: {gold_count} (expected 3984)")
assert gold_count == 3984, f"Expected 3984 gold rows, got {gold_count}"

display(gold_feedback_sentiment.limit(5))

gold_feedback_sentiment rows: 3984 (expected 3984)


event_date,difficulty_level,session_name,meeting_uuid,planned_duration_minutes,presenter,topic_category,sentiment,respondent_id,answer,question
2025-11-03,Intermediate,Serverless Compute Deep Dive,/e3hcn3IRDO17ugBv3bXHg==,60,Alex Martinez,DevOps,positive,ze8gFlIwQOasa1s6JVi+4w==,I think finally understand how all the pieces fit together end to end,What did you like about the content in this session?
2025-11-03,Intermediate,Serverless Compute Deep Dive,/e3hcn3IRDO17ugBv3bXHg==,60,Alex Martinez,DevOps,mixed,ze8gFlIwQOasa1s6JVi+4w==,"I have to say, the session assumed too much prior knowledge of the underlying infrastructure My only complaint is that it went by too fast.",What could have been improved?
2025-11-03,Intermediate,Serverless Compute Deep Dive,/e3hcn3IRDO17ugBv3bXHg==,60,Alex Martinez,DevOps,positive,So1HPj3vRCGQHXZf/eoAAg==,"To be frank, great real-world examples that connected theory to practice Looking forward to more sessions like this.",What did you like about the content in this session?
2025-11-03,Intermediate,Serverless Compute Deep Dive,/e3hcn3IRDO17ugBv3bXHg==,60,Alex Martinez,DevOps,positive,YAvpMZ8dQlebITLw9BDXRg==,The analogies used to explain the architecture were spot on Will be sharing the notebooks with my team.,What did you like about the content in this session?
2025-11-03,Intermediate,Serverless Compute Deep Dive,/e3hcn3IRDO17ugBv3bXHg==,60,Alex Martinez,DevOps,mixed,YAvpMZ8dQlebITLw9BDXRg==,"From my perspective, could use more documentation links for follow-up reading My only complaint is that it went by too fast.",What could have been improved?


In [0]:
target_table = f"{catalog}.{schema}.gold_feedback_sentiment"

gold_feedback_sentiment.write.mode("overwrite").saveAsTable(target_table)
print(f"Wrote {gold_count} rows to {target_table}")

Wrote 3984 rows to learn_no_code_etl_designer_puli_reddy.designer_meeting.gold_feedback_sentiment


In [0]:
# Verify the written table independently.
written = spark.table(target_table)

row_count = written.count()
expected_columns = [
    "event_date",
    "difficulty_level",
    "session_name",
    "meeting_uuid",
    "planned_duration_minutes",
    "presenter",
    "sentiment",
    "answer",
    "question",
]
missing_columns = [c for c in expected_columns if c not in written.columns]
empty_answers = written.filter(col("answer").isNull() | (col("answer") == "")).count()
null_session_attrs = written.filter(col("session_name").isNull()).count()

print(f"row count:        {row_count} (expected 3984)")
print(f"missing columns:  {missing_columns or 'none'}")
print(f"empty answers:    {empty_answers} (expected 0)")
print(f"rows missing session attributes: {null_session_attrs} (expected 0)")

assert row_count == 3984, f"Expected 3984 rows, got {row_count}"
assert not missing_columns, f"Missing columns: {missing_columns}"
assert empty_answers == 0, "Gold table still contains empty answers"
assert null_session_attrs == 0, "Some sentiment rows did not match a session"

print("All checks passed.")
display(written.groupBy("sentiment").count().orderBy("count", ascending=False))

row count:        3984 (expected 3984)
missing columns:  none
empty answers:    0 (expected 0)
rows missing session attributes: 0 (expected 0)
All checks passed.


sentiment,count
positive,1741
neutral,1074
mixed,951
negative,218
